# EDA — Maquinarias (tabla NORMALIZADA con LLM)

Este notebook analiza los datos después de la Fase B (DeepSeek), comparando el antes y después del rescate por IA.

> ⚠️ **Contenido generado/asistido por IA** (DeepSeek): `marca_norm`, `modelo_match` y `*_norm`. `modelo_flag` indica la confianza: `ok` (match exacto), `alias` (mapeo curado inferido — validar), `low`/`nomatch` (en revisión). Validar con experto antes de decisiones.


In [ ]:
# ============================================================
# ANÁLISIS POST-LLM — MAQUINARIA PESADA NORMALIZADA
# Comparación: Fase A (parser) vs Fase B (DeepSeek)
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

print("✅ Librerías cargadas")


## 1.  Cargar datos pre y post LLM

In [ ]:
outputs_dir = Path('../outputs')

# Buscar archivos
estructurados = sorted(outputs_dir.glob('*_estructurado.xlsx'))
normalizados = sorted(outputs_dir.glob('*_normalizado.xlsx'))

if not normalizados:
    raise FileNotFoundError("❌ No hay archivos _normalizado.xlsx. Ejecuta extract_llm.py primero.")

archivo_pre = estructurados[-1] if estructurados else None
archivo_post = normalizados[-1]

print(f"📂 ANTES (Fase A): {archivo_pre.name if archivo_pre else 'No disponible'}")
print(f"📂 DESPUÉS (Fase B): {archivo_post.name}")

# Cargar datos normalizados
df_post = pd.read_excel(archivo_post, sheet_name='normalizado_llm')

# Cargar revisión si existe
try:
    df_revisar = pd.read_excel(archivo_post, sheet_name='_revisar_llm')
    tiene_revisar = True
except:
    df_revisar = pd.DataFrame()
    tiene_revisar = False

# Cargar vocab nuevo si existe
try:
    df_vocab = pd.read_excel(archivo_post, sheet_name='_vocab_nuevo')
    tiene_vocab = True
except:
    df_vocab = pd.DataFrame()
    tiene_vocab = False

print(f"✅ {len(df_post):,} registros normalizados cargados")
print(f"✅ Columnas: {len(df_post.columns)}"))


## 2. Impacto del LLM - Marcas rescatadas


In [ ]:
print("=" * 70)
print("🏷️  IMPACTO DEL LLM — MARCAS")
print("=" * 70)

# Marcas detectadas por el LLM
marcas_llm = df_post[df_post['marca_raw_llm'].notna()]

if len(marcas_llm) > 0:
    print(f"\n📊 El LLM procesó {len(marcas_llm):,} registros")
    print(f"   Marcas en vocabulario:     {marcas_llm['marca_in_vocab'].sum():,}")
    print(f"   Marcas NUEVAS (fuera):     {(~marcas_llm['marca_in_vocab']).sum():,}")
    
    # Top marcas nuevas encontradas
    if tiene_vocab and len(df_vocab) > 0:
        print(f"\n🆕 TOP MARCAS NUEVAS (para agregar al diccionario):")
        for _, row in df_vocab.head(10).iterrows():
            print(f"   {row['marca_norm']:<30} {int(row['unidades']):>4} unidades")
            if pd.notna(row.get('sugerencia')):
                print(f"   └─ Sugerencia: {row['sugerencia']}")
else:
    print("   (No se encontraron datos de LLM en este archivo)")

## 3. Calidad del modelo detectado


In [ ]:
print("\n" + "=" * 70)
print("🔍 CALIDAD DE DETECCIÓN DE MODELOS")
print("=" * 70)

# Flags de modelo
if 'modelo_flag' in df_post.columns:
    flags = df_post['modelo_flag'].value_counts()
    
    print(f"\n   Distribución de modelo_flag:")
    for flag, count in flags.items():
        pct = count / len(df_post) * 100
        bar = '█' * int(pct / 2)
        emoji = {'ok': '✅', 'alias': '🔄', 'low': '⚠️', 'nomatch': '❌', 'sin_dato': '⬜', 'sin_respuesta': '💤'}
        print(f"   {emoji.get(flag, '❓')} {flag:<15} {count:>6} ({pct:>5.1f}%)  {bar}")

# Score de confianza
if 'modelo_score' in df_post.columns:
    scores = df_post['modelo_score'].dropna()
    print(f"\n   Score promedio: {scores.mean():.1f}%")
    print(f"   Score mediana:  {scores.median():.1f}%")
    
    fig = px.histogram(
        scores, 
        nbins=20,
        title='📊 Distribución de Scores de Modelo (Fuzzy Match)',
        labels={'value': 'Score (%)', 'count': 'Frecuencia'}
    )
    fig.add_vline(x=90, line_dash="dash", line_color="green", annotation_text="OK (90%)")
    fig.add_vline(x=75, line_dash="dash", line_color="orange", annotation_text="Low (75%)")
    fig.show()

## 4. Enums validados


In [ ]:
print("\n" + "=" * 70)
print("✅ VALIDACIÓN DE ENUMS")
print("=" * 70)

# Validaciones de maquinaria
validaciones = ['tren_rodaje_valido', 'combustible_valido', 'categoria_maquinaria_valido']
nombres = ['Tren de Rodaje', 'Combustible', 'Categoría Maquinaria']

for val, nom in zip(validaciones, nombres):
    if val in df_post.columns:
        total = len(df_post)
        validos = df_post[val].sum()
        invalidos = total - validos
        pct = validos / total * 100
        print(f"\n   {nom}:")
        print(f"   ✅ Válidos:    {validos:>6} ({pct:>5.1f}%)")
        print(f"   ❌ Inválidos:  {invalidos:>6} ({100-pct:>5.1f}%)")
        
        # Mostrar valores inválidos como muestra
        if invalidos > 0:
            invalidos_df = df_post[~df_post[val]]
            col_norm = val.replace('_valido', '_norm')
            if col_norm in df_post.columns:
                muestra = invalidos_df[col_norm].dropna().value_counts().head(5)
                if len(muestra) > 0:
                    print(f"   📋 Valores problemáticos:")
                    for v, c in muestra.items():
                        print(f"      '{v}': {c} registros")


## 5. Registros a revisar


In [ ]:
print("\n" + "=" * 70)
print("🔍 REGISTROS QUE REQUIEREN REVISIÓN")
print("=" * 70)

if tiene_revisar and len(df_revisar) > 0:
    print(f"\n   Total a revisar: {len(df_revisar):,} registros")
    print(f"   ({len(df_revisar)/len(df_post)*100:.1f}% del total)")
    
    # ¿Por qué están en revisar?
    print(f"\n   📋 Motivos de revisión:")
    
    motivos = {
        'Modelo dudoso (low/nomatch)': df_revisar['modelo_flag'].isin(['low', 'nomatch']).sum() if 'modelo_flag' in df_revisar.columns else 0,
        'Alias de modelo': (df_revisar['modelo_flag'] == 'alias').sum() if 'modelo_flag' in df_revisar.columns else 0,
        'Marca fuera de vocab': (~df_revisar['marca_in_vocab']).sum() if 'marca_in_vocab' in df_revisar.columns else 0,
    }
    for motivo, count in motivos.items():
        if count > 0:
            print(f"   • {motivo}: {count:,}")
    
    # Muestra de registros a revisar
    print(f"\n   📄 MUESTRA (primeros 10):")
    cols_mostrar = ['marca_raw_llm', 'marca_norm', 'modelo_raw_llm', 'modelo_flag']
    cols_disponibles = [c for c in cols_mostrar if c in df_revisar.columns]
    if cols_disponibles:
        print(df_revisar[cols_disponibles].head(10).to_string(index=False))
else:
    print("\n   ✅ No hay registros en _revisar_llm")

## 6. Comparación ANTES vs DESPUÉS


In [ ]:
# Esta celda solo funciona si tienes ambos archivos
if archivo_pre and archivo_pre.exists():
    print("\n" + "=" * 70)
    print("📊 COMPARACIÓN: FASE A (Parser) vs FASE B (Parser + LLM)")
    print("=" * 70)
    
    df_pre = pd.read_excel(archivo_pre, sheet_name='estructurado')
    
    # Comparar marcas
    marcas_pre = df_pre['marca'].nunique()
    marcas_post = df_post['marca_norm'].nunique() if 'marca_norm' in df_post.columns else df_post['marca'].nunique()
    
    # Comparar modelos
    modelos_pre = df_pre['modelo'].nunique()
    modelos_post = df_post['modelo_match'].nunique() if 'modelo_match' in df_post.columns else df_post['modelo'].nunique()
    
    print(f"\n   {'Métrica':<30} {'ANTES':>10} {'DESPUÉS':>10} {'MEJORA':>10}")
    print(f"   {'-'*60}")
    print(f"   {'Marcas distintas':<30} {marcas_pre:>10} {marcas_post:>10} {marcas_post-marcas_pre:>+10}")
    print(f"   {'Modelos distintos':<30} {modelos_pre:>10} {modelos_post:>10} {modelos_post-modelos_pre:>+10}")
    print(f"   {'Registros procesados':<30} {len(df_pre):>10} {len(df_post):>10}")
    
    # Gráfico comparativo
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name='Fase A (Parser)',
        x=['Marcas', 'Modelos'],
        y=[marcas_pre, modelos_pre],
        marker_color='#1F4E79'
    ))
    fig.add_trace(go.Bar(
        name='Fase B (Parser + LLM)',
        x=['Marcas', 'Modelos'],
        y=[marcas_post, modelos_post],
        marker_color='#2E86C1'
    ))
    fig.update_layout(
        title='📊 ANTES vs DESPUÉS: Marcas y Modelos Detectados',
        barmode='group',
        height=400
    )
    fig.show()
else:
    print("\n⚠️  No se encontró archivo _estructurado.xlsx para comparar")

## 7. Costo estimado de la Fase B

In [ ]:
print("\n" + "=" * 70)
print("💰 COSTO ESTIMADO DE LA FASE B (LLM)")
print("=" * 70)

# Contar registros procesados por LLM
if 'fuente' in df_post.columns:
    llm_registros = df_post[df_post['fuente'].str.contains('LLM', na=False)]
    total_llm = len(llm_registros)
    
    # Estimación (DeepSeek ~$0.001 por registro)
    costo_estimado = total_llm * 0.001
    
    print(f"\n   Registros procesados por LLM: {total_llm:,}")
    print(f"   Costo estimado (DeepSeek):   ${costo_estimado:.2f} USD")
    print(f"\n   💡 Sin caché, el costo sería: ${costo_estimado:.2f}")
    print(f"   💾 Con caché (re-ejecuciones): $0.00 (gratis)")


## 8.  Resumen ejecutivo


In [ ]:
print("\n" + "=" * 70)
print("📋 RESUMEN EJECUTIVO")
print("=" * 70)

print(f"""
   📂 Archivo: {archivo_post.name}
   📊 Total registros: {len(df_post):,}
   
   🏷️  MARCAS:
      • En vocabulario:  {df_post['marca_in_vocab'].sum() if 'marca_in_vocab' in df_post.columns else 'N/D':>10}
      • Nuevas (fuera):  {(~df_post['marca_in_vocab']).sum() if 'marca_in_vocab' in df_post.columns else 'N/D':>10}
   
   🔍 MODELOS:
      • OK (≥90%):       {(df_post['modelo_flag']=='ok').sum() if 'modelo_flag' in df_post.columns else 'N/D':>10}
      • Low (≥75%):      {(df_post['modelo_flag']=='low').sum() if 'modelo_flag' in df_post.columns else 'N/D':>10}
      • No match:        {(df_post['modelo_flag']=='nomatch').sum() if 'modelo_flag' in df_post.columns else 'N/D':>10}
   
   ⚠️  REGISTROS A REVISAR: {len(df_revisar) if tiene_revisar else 0:,}
   💰 COSTO LLM: ${costo_estimado:.2f}
   
   ✅ Pipeline completado exitosamente
""")

print("🚀 Análisis post-LLM finalizado.")


## Cierre

- Tabla normalizada lista para análisis de mercado por **marca/modelo canónicos** y specs estandarizadas.
- Las filas `alias` (mapeo inferido) y `low`/`nomatch` están en la hoja `_revisar_llm` para validación de experto.
- Re-normalizar tras ajustar el diccionario (`data/vocab_extra.json`) es gratis (cache del LLM).
